# KrishiBid — crop disease classifier

Fine-tunes **MobileNetV3-Small** on a documented PlantVillage subset, then hands off to `export_onnx.py`.

**Run this on Colab** (free T4) — training is ~10 minutes; the rest of the project never needs Python.

### Deliberate scoping
10 classes across 3 crops relevant to Bangladesh (rice, potato, tomato) rather than PlantVillage's full 38. A model that covers a documented subset honestly is worth more than one that claims broad coverage it cannot support.

### The limitation to state up front
PlantVillage images are lab-captured: single leaf, uniform background, even lighting. Field photos are none of those things, so **real-world accuracy will be materially lower than the number this notebook reports.** That is why the server withholds any diagnosis below `DISEASE_CONFIDENCE_THRESHOLD` (0.60) and refers the farmer to an extension officer instead. Saying this before being asked is more credible than being caught by it.

In [ ]:
import json, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_SIZE, BATCH_SIZE, EPOCHS, LR = 224, 32, 12, 3e-4
PATIENCE = 3  # early stopping

ARTIFACTS = Path('../artifacts'); ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(DEVICE, torch.__version__)

## 1. Data

Expects `ml/data/<class_name>/*.jpg`, where class names match `crop__disease`:

```
ml/data/rice__brown_spot/…
ml/data/potato__late_blight/…
```

Get PlantVillage from Kaggle (`vipoooool/new-plant-diseases-dataset`), keep only the target classes, and rename folders to that convention. `ml/data/` is gitignored — the dataset is several GB and does not belong in the repo.

In [ ]:
DATA_DIR = Path('../data')

# Augmentation models real-world capture variance: a farmer's photo is rotated,
# off-centre, and lit differently from a lab plate. Without this the model overfits
# to PlantVillage's uniform framing and collapses on real input.
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Eval transform must match the server's preprocessing in
# server/src/services/diagnosis.service.ts exactly — same resize, same normalisation.
# A mismatch here silently degrades production accuracy with no visible error.
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full = datasets.ImageFolder(DATA_DIR, transform=train_tf)
print(f'{len(full)} images, {len(full.classes)} classes')
for name, idx in sorted(full.class_to_idx.items(), key=lambda kv: kv[1]):
    print(f'  [{idx}] {name}')

In [ ]:
# 70/15/15. Val and test get the eval transform (no augmentation) — augmenting the
# held-out sets would report a number that has nothing to do with deployment.
n_total = len(full)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

gen = torch.Generator().manual_seed(SEED)
train_ds, val_ds, test_ds = random_split(full, [n_train, n_val, n_test], generator=gen)

for subset in (val_ds, test_ds):
    subset.dataset = datasets.ImageFolder(DATA_DIR, transform=eval_tf)

loaders = {
    'train': DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True),
    'val':   DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2),
    'test':  DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2),
}
print({k: len(v.dataset) for k, v in loaders.items()})

## 2. Model

**MobileNetV3-Small**, chosen for the deployment constraint rather than for peak accuracy: ~2.5 M parameters, ~2.5 MB once INT8-quantised, and CPU inference in tens of milliseconds. A ResNet50 would score a little higher and would not fit the 512 MB free dyno alongside Node.

In [ ]:
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(full.classes))
model = model.to(DEVICE)

# Class weights: PlantVillage is imbalanced, and unweighted loss would let the model
# score well by neglecting the rarest disease — which is often the one worth catching.
counts = np.bincount([full.targets[i] for i in train_ds.indices], minlength=len(full.classes))
weights = torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=weights.to(DEVICE), label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print(f'trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = correct = seen = 0
    with torch.set_grad_enabled(train):
        for images, targets in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            if train:
                loss.backward(); optimizer.step()
            total_loss += loss.item() * targets.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
            seen += targets.size(0)
    return total_loss / seen, correct / seen

best_val, stale = 0.0, 0
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(loaders['train'], True)
    va_loss, va_acc = run_epoch(loaders['val'], False)
    scheduler.step()
    print(f'epoch {epoch:2d}  train {tr_acc:.4f}  val {va_acc:.4f}  ({time.time()-t0:.0f}s)')

    if va_acc > best_val:
        best_val, stale = va_acc, 0
        # class_to_idx is saved WITH the weights — export_onnx.py derives labels.json
        # from it, which is what keeps logit order and label order in agreement.
        torch.save({'state_dict': model.state_dict(),
                    'class_to_idx': full.class_to_idx,
                    'val_acc': va_acc}, ARTIFACTS / 'best.pt')
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f'early stop (no val improvement in {PATIENCE} epochs)')
            break

print(f'best val accuracy: {best_val:.4f}')

## 3. Evaluate on the held-out test split

Reported honestly: accuracy alone hides a model that ignores a rare class, so macro-F1, per-class precision/recall and the confusion matrix all go into `artifacts/metrics.json` and the README.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

model.load_state_dict(torch.load(ARTIFACTS / 'best.pt')['state_dict'])
model.eval()

y_true, y_pred, y_conf = [], [], []
with torch.no_grad():
    for images, targets in loaders['test']:
        probs = torch.softmax(model(images.to(DEVICE)), dim=1).cpu()
        y_true += targets.tolist()
        y_pred += probs.argmax(1).tolist()
        y_conf += probs.max(1).values.tolist()

labels = [n for n, _ in sorted(full.class_to_idx.items(), key=lambda kv: kv[1])]
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True, zero_division=0)

accuracy = report['accuracy']
macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f'test accuracy {accuracy:.4f}   macro-F1 {macro_f1:.4f}')
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

# How much of the test set the server would refuse to diagnose at threshold 0.60.
# This is the number that tells you whether the abstention rule is usable or whether
# it would silently reject most real queries.
THRESHOLD = 0.60
conf = np.array(y_conf)
abstained = float((conf < THRESHOLD).mean())
confident = conf >= THRESHOLD
acc_confident = float((np.array(y_true)[confident] == np.array(y_pred)[confident]).mean()) if confident.any() else 0.0
print(f'\nabstention rate @{THRESHOLD}: {abstained:.1%}')
print(f'accuracy on confident predictions: {acc_confident:.4f}')

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('predicted'); ax.set_ylabel('actual')
ax.set_title(f'Confusion matrix — acc {accuracy:.3f}, macro-F1 {macro_f1:.3f}')
for i in range(len(labels)):
    for j in range(len(labels)):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=6)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'confusion-matrix.png', dpi=140)
plt.show()

metrics = {
    'modelVersion': 'v1-mobilenetv3s',
    'architecture': 'mobilenet_v3_small',
    'dataset': 'PlantVillage subset (rice, potato, tomato)',
    'split': {'train': n_train, 'val': n_val, 'test': n_test},
    'testAccuracy': round(accuracy, 4),
    'macroF1': round(macro_f1, 4),
    'confidenceThreshold': THRESHOLD,
    'abstentionRate': round(abstained, 4),
    'accuracyOnConfident': round(acc_confident, 4),
    'perClass': {
        name: {k: round(v, 4) for k, v in report[name].items()} for name in labels
    },
    'knownLimitation': (
        'PlantVillage is lab-imaged (single leaf, uniform background). Field accuracy '
        'will be materially lower; the API withholds any prediction below the '
        'confidence threshold and refers the farmer to an extension officer.'
    ),
}
(ARTIFACTS / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('wrote artifacts/metrics.json')

## 4. Export

```bash
cd ml
python export_onnx.py --checkpoint artifacts/best.pt --version v1
```

Then point `DISEASE_MODEL_PATH` at `../ml/artifacts/model-v1.onnx` and restart the server. `GET /api/diagnosis/health` should report `ready: true`.